# Tutorial on how to create an agent using Langchain

## Importing API keys: Langsmith and Tavily search engine

In [1]:
import getpass
import os

os.environ["LANGSMITH_TRACING"]="true"
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter API Key for TAVILY: ")
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()
# os.environ["TAVILY_API_KEY"] = getpass.getpass()

## Calling Tavily Search Engine to do a search based on user's query

In [2]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(max_results=2)
search_results = search.invoke("What is the weather like in Dubai?")
print(search_results)

tools = [search]

[{'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Dubai', 'region': 'Dubai', 'country': 'United Arab Emirates', 'lat': 25.2522, 'lon': 55.28, 'tz_id': 'Asia/Dubai', 'localtime_epoch': 1739284330, 'localtime': '2025-02-11 18:32'}, 'current': {'last_updated_epoch': 1739284200, 'last_updated': '2025-02-11 18:30', 'temp_c': 23.1, 'temp_f': 73.6, 'is_day': 0, 'condition': {'text': 'Clear', 'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png', 'code': 1000}, 'wind_mph': 10.3, 'wind_kph': 16.6, 'wind_degree': 16, 'wind_dir': 'NNE', 'pressure_mb': 1016.0, 'pressure_in': 30.0, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 53, 'cloud': 0, 'feelslike_c': 25.0, 'feelslike_f': 77.0, 'windchill_c': 20.9, 'windchill_f': 69.6, 'heatindex_c': 20.9, 'heatindex_f': 69.6, 'dewpoint_c': 13.4, 'dewpoint_f': 56.0, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 0.0, 'gust_mph': 15.9, 'gust_kph': 25.6}}"}, {'url': 'https://www.peoplesweather.com/weather/Dubai/?date=2025-02-11', 'conte

## Using a language model (GROQ)

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

### import groq model

In [4]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter API Key for GROQ: ")

model = init_chat_model("llama3-8b-8192", model_provider='groq')

### Testing the model with a simple input query

In [5]:
response = model.invoke([HumanMessage(content="hi")])
print(response.content)

Hi! It's nice to meet you. Is there something I can help you with or would you like to chat?


### Enable the model to do tool calling

#### if we invoke the model with a simple query that is not related to the tool, then it gives a simple anwser and does not invoke any tools

In [16]:
model_with_tools = model.bind_tools(tools)
response = model_with_tools.invoke([HumanMessage(content="What is the height of Eiffel Tower")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': 'What is the height of Eiffel Tower?'}, 'id': 'call_9kq0', 'type': 'tool_call'}]


## Creating the Agent

### Since we have the LLM and the tool,  we now will use langgraph to create the agent.

In [21]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(model, tools)

response = agent_executor.invoke({"messages": [HumanMessage(content="Use the tool")]})

response["messages"]

[HumanMessage(content='Use the tool', additional_kwargs={}, response_metadata={}, id='68c1e2b6-cfa8-4b5c-99f6-6b42e8c12ebb'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_1d87', 'function': {'arguments': '{"query":"What is the definition of a comprehensive, accurate, and trusted results search engine?"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 942, 'total_tokens': 1033, 'completion_time': 0.075833333, 'prompt_time': 0.126438786, 'queue_time': 0.023557982000000005, 'total_time': 0.202272119}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_179b0f92c9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-d2ed47f4-e21e-470a-b9f7-e3ce96be323d-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'What is the definition of a comprehensive, accurate, and trusted results search engine?'}, 'id': 'call_1d87', 'type': 'tool_call'}], usage

### This agent is stateless which means it cannot remember the previous interactions so we will add checkpointer.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent_executor = create_react_agent(model, tools, checkpointer=memory)
config = {
    "configurable": {
    "thread_id": "abc123"
}}

